# Phase 5 — Sensitivity Analysis and Report

Runs documented scenarios and exports report-ready comparison tables.

In [ ]:
from pathlib import Path
import sys
import geopandas as gpd
import pandas as pd
ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.config import PROCESSED_DIR, TABLES_DIR
from src.analysis.conversion_scoring import load_data, load_planned, prepare_candidate_base
from src.analysis.phase5_delivery import default_scenarios, run_sensitivity, summarize_candidate_stability, severity_sensitivity, scenario_metadata

# Phase 3 tract metrics (pre-scoring) are re-scored under every scenario
metrics = gpd.read_parquet(PROCESSED_DIR / 'tract_coverage_metrics.parquet')
gas, ev, _, highways = load_data()
# Scenario-invariant candidate inputs: nearest DCFC, nearest highway, planned DCFC penalty
candidate_base = prepare_candidate_base(gas, ev, highways, load_planned())
print('Tracts:', len(metrics), 'Candidate gas stations:', len(candidate_base))

In [ ]:
scenarios = default_scenarios()
scenario_metadata(scenarios).to_csv(TABLES_DIR / 'sensitivity_scenarios.csv', index=False)
tract_results, ranked, skipped = run_sensitivity(metrics, candidate_base, gas, scenarios)
print('Skipped scenarios:', skipped or 'none')
severity_summary = severity_sensitivity(tract_results)
severity_summary.to_csv(TABLES_DIR / 'severity_sensitivity.csv', index=False)
severity_summary

In [ ]:
# Each scenario's candidates were re-scored against that scenario's tract scores
stability = summarize_candidate_stability(ranked)
stability.to_csv(TABLES_DIR / 'candidate_stability.csv', index=False)
print('Stable candidates:', int(stability['stable_candidate'].sum()))
stability.head(20)

## Interpretation checklist

Confirm baseline consistency, report severity changes, identify stable candidates, and describe results as model-based prioritization rather than feasibility determinations.